# GENERAR COTIZACION

In [1]:
from processor.extractor import extract

fs = "pruebas/Transversal 56 # 104b-33_260331_172537 (1).pdf"

content = extract(fs)

In [2]:
print(content.text)

305 | | ||| ]
Maria Paula Arevalo


.5+20
— Total 27m
2.300.000



In [3]:
print(len(content.images))

4


In [ ]:
from IPython.display import display

def show_images(images, max_width=400):
    """Muestra una lista de imágenes PIL en el notebook."""
    for i, img in enumerate(images):
        print(f"Imagen {i + 1} ({img.size[0]}x{img.size[1]})")
        img_copy = img.copy()
        if img_copy.width > max_width:
            ratio = max_width / img_copy.width
            img_copy = img_copy.resize((max_width, int(img_copy.height * ratio)))
        display(img_copy)

show_images(content.images)

In [4]:
import json
import tempfile
from pathlib import Path


def flatten_cotization(config_path: str = "config/cotization.json") -> str:
    """Lee cotization.json (formato productos) y genera un archivo temporal
    con formato plano compatible con extract_data ({"fields": [...], "models": {...}}).
    Los campos se prefijan con el id del producto: {id}_{campo}.
    Retorna la ruta del archivo temporal."""
    with open(config_path, encoding="utf-8") as f:
        config = json.load(f)

    flat_fields = []
    for producto in config["productos"]:
        pid = producto["id"]
        for field in producto["fields"]:
            flat_field = {**field, "name": f"{pid}_{field['name']}"}
            if "description" not in flat_field:
                flat_field["description"] = f"{producto['descripcion']} — {field['name']}"
            flat_fields.append(flat_field)

    flat_config = {"fields": flat_fields, "models": config["models"]}

    fd, tmp_path = tempfile.mkstemp(suffix=".json", prefix="cotization_flat_")
    tmp = Path(tmp_path)
    tmp.write_text(json.dumps(flat_config, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(tmp)


def parse_price(price_flat: dict, config_path: str = "config/cotization.json") -> dict:
    """Reagrupa el dict plano del LLM en sub-dicts por producto.
    Retorna: {"cotizacion": {"costo_total": ..., "metros": ...}, "tiene_wallbox": {...}, ...}
    """
    with open(config_path, encoding="utf-8") as f:
        config = json.load(f)

    result = {}
    for producto in config["productos"]:
        pid = producto["id"]
        sub = {}
        for field in producto["fields"]:
            key = f"{pid}_{field['name']}"
            sub[field["name"]] = price_flat.get(key)
        result[pid] = sub
    return result

In [5]:
from llm_client.client import extract_data
import json

data = extract_data(
    text=content.text,
    file_path=content.source,
    provider="minimax",
)

flat_config_path = flatten_cotization()
price_flat = extract_data(
    text=content.text,
    file_path=content.source,
    provider="minimax",
    data_path=flat_config_path,
    prompt_path="config/prompt_cotizacion.md",
)
Path(flat_config_path).unlink(missing_ok=True)

price = parse_price(price_flat)

print(json.dumps(data, ensure_ascii=False, indent=2))
print(json.dumps(price, ensure_ascii=False, indent=2))

/home/jrsbot/work/Documents/pluguea/informes/linker/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


{
  "nombre": "Maria Paula Arevalo",
  "is_male": false,
  "direccion": "Transversal 56 # 104b-33",
  "apartamento": null,
  "nombre_edificio": null,
  "ciudad_departamento": "Bogotá D.C.",
  "tipo_vehiculo": "Tesla"
}
{
  "cotizacion": {
    "costo_total": 2300000,
    "metros": 27
  },
  "tiene_nema_14_50": {
    "incluido": false,
    "costo": 250000
  },
  "tiene_wallbox": {
    "incluido": false,
    "costo": 250000
  }
}


In [6]:
class Product:

    id = 0

    def __init__(self, qty : int, desc : str, unit : str, vr_uni_inc :  int):
        self.item = Product.id + 1
        Product.id += 1
        self.qty = qty
        self.desc = desc
        self.unit = unit
        self.vr_uni_inc = vr_uni_inc
        self.set_iva()
        self.set_vr_total()

    def set_iva(self):
        self.iva = self.vr_uni_inc * 0.19

    def set_vr_total(self):
        self.vr_total = self.vr_uni_inc + self.iva

    def get_list(self):
        return [self.qty, self.desc, self.unit, self.vr_uni_inc, self.iva, self.vr_total]

    @property
    def um(self) -> str:
        return self.unit

    @property
    def value(self) -> int:
        return self.vr_uni_inc

    def __str__(self):
        return f"Product:\n(item={self.item},\ncantidad={self.qty},\ndescripcion='{self.desc}',\nun_medida='{self.unit}',\nvr_uni_inc={self.vr_uni_inc},\niva={self.iva},\nvr_total={self.vr_total})"

In [7]:
import json

with open("config/producs.json") as f:
    _producs = json.load(f)


In [8]:
_p_cotizacion = next(p for p in _producs if p["id"] == "cotizacion")

cotizacion = price.get("cotizacion", {})
if cotizacion.get("costo_total"):
    product = Product(
        cotizacion["metros"],
        _p_cotizacion["descripción"],
        _p_cotizacion["un. medida"],
        cotizacion["costo_total"],
    )
else:
    product = Product(
        _p_cotizacion["cantidad"] or 0,
        _p_cotizacion["descripción"],
        _p_cotizacion["un. medida"],
        _p_cotizacion["valor unitario"] or 0,
    )

products = [product]
print(product)

Product:
(item=1,
cantidad=27,
descripcion='Instalación de estación de carga incluido cable de cobre, tubería EMT, Protección Termo Magnética, Protección diferencial Caja Mini incluye tesla nema 14-50',
un_medida='metros',
vr_uni_inc=2300000,
iva=437000.0,
vr_total=2737000.0)


# GENERAR DOCUMENTO

In [ ]:
import math
from docxtpl import DocxTemplate, RichText
from datetime import date
import re
import tempfile
import fcntl
import json
from pathlib import Path
from docx import Document

_MESES = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre",
]

def _fmt(n) -> str:
    """Formatea un número como entero con separador de miles (.)."""
    return f"{int(n):,}".replace(",", ".")


_IF_BLOCK = re.compile(r'\{%-?\s*if\s+\w+\s*-?%\}.*\{%-?\s*endif\s*-?%\}', re.DOTALL)
_SENTINEL = "\u00abCOND\u00bb"  # «COND» — marca párrafos condicionales

CONFIG_PATH = Path("config/data.json")


def _increment_quote_number() -> str:
    """Atomically reads and increments the quote number in config/data.json.
    Uses file locking to prevent race conditions."""
    lock_path = CONFIG_PATH.with_suffix(".lock")
    with open(lock_path, "w") as lockf:
        fcntl.flock(lockf.fileno(), fcntl.LOCK_EX)
        try:
            with open(CONFIG_PATH, "r") as f:
                config = json.load(f)
            current = int(config.get("number", "0"))
            new_number = str(current + 1)
            config["number"] = new_number
            tmp_path = CONFIG_PATH.with_suffix(".tmp")
            with open(tmp_path, "w") as f:
                json.dump(config, f, indent=2)
            Path(tmp_path).replace(CONFIG_PATH)
            return new_number
        finally:
            fcntl.flock(lockf.fileno(), fcntl.LOCK_UN)


def _prepare_template(template_path: str) -> str:
    """Copia el template a un temporal añadiendo el sentinel al inicio
    de cada párrafo condicional {% if %}...{% endif %}.
    Retorna la ruta del archivo temporal."""
    doc = Document(str(template_path))
    for para in doc.paragraphs:
        if _IF_BLOCK.search(para.text) and para.runs:
            para.runs[0].text = _SENTINEL + para.runs[0].text
    fd, tmp = tempfile.mkstemp(suffix=".docx")
    import os; os.close(fd)
    doc.save(tmp)
    return tmp


def _remove_null_paragraphs(output_path: Path):
    """Elimina párrafos donde el sentinel quedó solo (condición False)
    y limpia el sentinel de los párrafos donde la condición fue True."""
    doc = Document(str(output_path))
    to_remove = []
    for para in doc.paragraphs:
        if para.text.strip() == _SENTINEL:
            to_remove.append(para)
        elif _SENTINEL in para.text:
            for run in para.runs:
                if _SENTINEL in run.text:
                    run.text = run.text.replace(_SENTINEL, "")
                    break
    for para in to_remove:
        para._element.getparent().remove(para._element)
    doc.save(str(output_path))


def fill_template(
    data: dict,
    products: list,
    template_path: str = "config/template_base.docx",
    output_dir: str = "pruebas",
    fotos: list[dict] | None = None,
) -> Path:
    """Rellena template_base.docx con los datos extraídos y guarda el resultado."""
    tmp_template = _prepare_template(template_path)

    try:
        tpl = DocxTemplate(tmp_template)

        hoy = date.today()
        fecha = f"{hoy.day} de {_MESES[hoy.month - 1]} de {hoy.year}"

        ciudad_dep = data.get("ciudad_departtamento") or data.get("ciudad_departamento") or "Bogotá D.C."
        ciudad = ciudad_dep.split(",")[0].strip()

        total_iva = sum(p.iva for p in products)
        valor_total = sum(p.vr_total for p in products)
        valor_inicial = math.floor(valor_total / 2 / 100_000) * 100_000
        number = _increment_quote_number()

        context = {
            "NOMBRE":              (data.get("nombre") or "").upper(),
            "is_male":             "Señor" if data.get("is_male", True) else "Señora",
            "direccion":           data.get("direccion", ""),
            "apartamento":         data.get("apartamento"),
            "nombre_edificio":     data.get("nombre_edificio"),
            "ciudad_departamento": ciudad_dep,
            "ciudad":              ciudad,
            "fecha":               fecha,
            "date":                hoy.strftime("%Y-%m-%d"),
            "tipo_vehiculo":       data.get("tipo_vehiculo") or "Tesla",
            "fotos":               fotos or [],
            "new_page":            RichText("\f"),
            "number":              number,
            "products":            [
                type("P", (), dict(
                    item=p.item, qty=p.qty, desc=p.desc, unit=p.unit,
                    value=_fmt(p.value), iva=_fmt(p.iva), vr_total=_fmt(p.vr_total)
                ))() for p in products
            ],
            "total_iva":           _fmt(total_iva),
            "valor_total":         _fmt(valor_total),
            "valor_inicial":        f"{valor_inicial:,}",
        }

        tpl.render(context)

        direccion_slug = re.sub(r'[^\w\s-]', '', data.get("direccion", "documento")).strip().replace(" ", "_")
        timestamp = hoy.strftime("%Y%m%d")
        output_path = Path(output_dir) / f"{direccion_slug}_{timestamp}.docx"
        tpl.save(output_path)
    finally:
        Path(tmp_template).unlink(missing_ok=True)

    _remove_null_paragraphs(output_path)
    return output_path

In [ ]:
print(data)
products[0].vr_total = 3568000
print(*products)

In [ ]:
output_file = fill_template(data, products)
print(f"Archivo generado: {output_file}")